### Table Structure Evaluation — OmniDocBench JASyn

Đánh giá 3 model Table Structure (**`TableFormer ACCURATE`**, **`RapidTable — SLANet_plus`**, **`RapidTable — UniTable`**) trên **354 trang có bảng (406 bảng tiếng Nhật)** của **OmniDocBench JASyn** (cố định Layout = `DocLayout-YOLO`, OCR = `PP-OCRv6_medium`, Dynamic Image Scaling; chuẩn hóa chữ Nhật `NFKC` + bỏ khoảng trắng thừa).

* **`TEDS` *(Tree-Edit-Distance-based Similarity)*:** Độ tương đồng cây HTML bảng trên cả cấu trúc (`<tr>`, `<td>`, `rowspan`, `colspan`) lẫn nội dung chữ trong ô.
* **`TEDS-Struct` *(Structure-only TEDS)*:** Chỉ đo độ chính xác thuần cấu trúc hàng/cột/gộp ô, tách khỏi sai số OCR.
* **`Matched %` & `Matched-only`:** Tỷ lệ bảng được bước Layout (`DocLayout-YOLO`) khoanh trúng và điểm `TEDS` / `TEDS-Struct` tính riêng trên các bảng khớp đó.

In [1]:
import json
import re
import unicodedata
from pathlib import Path

import pandas as pd
from IPython.display import display
from lxml import html as lxml_html

BASE_DIR = Path(".").resolve()
GT_PATH = BASE_DIR / "ground_truth.json"
MODELS = [
    "TableFormer ACCURATE",
    "RapidTable — SLANet_plus",
    "RapidTable — UniTable",
]


def levenshtein(a: str, b: str) -> int:
    if a == b:
        return 0
    if not a:
        return len(b)
    if not b:
        return len(a)
    if len(a) > len(b):
        a, b = b, a
    m = len(a)
    peq: dict[str, int] = {}
    for i, ch in enumerate(a):
        peq[ch] = peq.get(ch, 0) | (1 << i)
    pv = (1 << m) - 1
    mv = 0
    score = m
    hb = 1 << (m - 1)
    mask = (1 << m) - 1
    for ch in b:
        eq = peq.get(ch, 0)
        xv = eq | mv
        xh = (((eq & pv) + pv) ^ pv) | eq
        ph = mv | (~(xh | pv) & mask)
        mh = pv & xh
        if ph & hb:
            score += 1
        elif mh & hb:
            score -= 1
        ph = ((ph << 1) | 1) & mask
        mh = (mh << 1) & mask
        pv = mh | (~(xv | ph) & mask)
        mv = ph & xv
    return score


def norm_cell_text_ja(s: str) -> str:
    s = unicodedata.normalize("NFKC", s or "")
    return re.sub(r"\s+", "", s)


class Node:
    __slots__ = ("tag", "colspan", "rowspan", "text", "children")

    def __init__(self, tag, colspan=1, rowspan=1, text="", children=None):
        self.tag = tag
        self.colspan = colspan
        self.rowspan = rowspan
        self.text = text
        self.children = children if children is not None else []


def html_to_tree(html_str: str) -> Node:
    if not html_str or not html_str.strip():
        return Node("table")
    try:
        doc = lxml_html.fromstring(html_str)
    except Exception:
        return Node("table")
    tables = doc.xpath(".//table") if doc.tag.lower() != "table" else [doc]
    tbl = tables[0] if tables else doc
    rows = []
    for tr in tbl.xpath(".//tr"):
        cells = []
        for cell in tr.xpath("./td | ./th"):
            try:
                cs = max(1, int(cell.attrib.get("colspan", 1)))
            except Exception:
                cs = 1
            try:
                rs = max(1, int(cell.attrib.get("rowspan", 1)))
            except Exception:
                rs = 1
            txt = norm_cell_text_ja("".join(cell.itertext()))
            cells.append(Node("td", cs, rs, txt))
        if cells:
            rows.append(Node("tr", children=cells))
    return Node("table", children=rows)


def docling_table_to_tree(tbl: dict) -> Node:
    data = tbl.get("data") or {}
    cells = data.get("table_cells") or []
    if not cells:
        return Node("table")
    by_row: dict[int, list[tuple[int, Node]]] = {}
    for c in cells:
        r0 = int(c.get("start_row_offset_idx", 0))
        c0 = int(c.get("start_col_offset_idx", 0))
        rs = max(1, int(c.get("row_span", 1)))
        cs = max(1, int(c.get("col_span", 1)))
        txt = norm_cell_text_ja(c.get("text", ""))
        by_row.setdefault(r0, []).append((c0, Node("td", cs, rs, txt)))
    rows = []
    for r_idx in sorted(by_row):
        row_cells = [n for _, n in sorted(by_row[r_idx], key=lambda x: x[0])]
        rows.append(Node("tr", children=row_cells))
    return Node("table", children=rows)


def is_complex_tree(root: Node) -> bool:
    return any(c.colspan > 1 or c.rowspan > 1 for tr in root.children for c in tr.children)


def tree_size(root: Node) -> int:
    return 1 + sum(1 + len(tr.children) for tr in root.children)


def cell_rename_cost(a: Node, b: Node, struct_only: bool) -> float:
    if a.colspan != b.colspan or a.rowspan != b.rowspan:
        return 1.0
    if struct_only or a.text == b.text:
        return 0.0
    denom = max(len(a.text), len(b.text))
    return levenshtein(a.text, b.text) / denom if denom > 0 else 0.0


def row_ted(ra: Node, rb: Node, struct_only: bool) -> float:
    ca, cb = ra.children, rb.children
    m, n = len(ca), len(cb)
    prev = [float(j) for j in range(n + 1)]
    for i in range(1, m + 1):
        curr = [float(i)] + [0.0] * n
        ai = ca[i - 1]
        for j in range(1, n + 1):
            cost = cell_rename_cost(ai, cb[j - 1], struct_only)
            curr[j] = min(prev[j] + 1.0, curr[j - 1] + 1.0, prev[j - 1] + cost)
        prev = curr
    return prev[n]


def compute_teds(pred: Node, gt: Node, struct_only: bool = False) -> float:
    sa, sb = tree_size(pred), tree_size(gt)
    if sa == 1 and sb == 1:
        return 1.0
    if sa == 1 or sb == 1:
        return 0.0
    ra, rb = pred.children, gt.children
    m, n = len(ra), len(rb)
    prev = [0.0] * (n + 1)
    for j in range(1, n + 1):
        prev[j] = prev[j - 1] + 1.0 + len(rb[j - 1].children)
    for i in range(1, m + 1):
        del_cost = 1.0 + len(ra[i - 1].children)
        curr = [prev[0] + del_cost] + [0.0] * n
        rai = ra[i - 1]
        for j in range(1, n + 1):
            ins_cost = 1.0 + len(rb[j - 1].children)
            sub_cost = row_ted(rai, rb[j - 1], struct_only)
            curr[j] = min(prev[j] + del_cost, curr[j - 1] + ins_cost, prev[j - 1] + sub_cost)
        prev = curr
    return max(0.0, 1.0 - prev[n] / max(sa, sb))


def box_iou(b1, b2) -> float:
    ix0, iy0 = max(b1[0], b2[0]), max(b1[1], b2[1])
    ix1, iy1 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0.0, ix1 - ix0) * max(0.0, iy1 - iy0)
    if inter <= 0:
        return 0.0
    a1 = max(0.0, b1[2] - b1[0]) * max(0.0, b1[3] - b1[1])
    a2 = max(0.0, b2[2] - b2[0]) * max(0.0, b2[3] - b2[1])
    return inter / (a1 + a2 - inter) if (a1 + a2 - inter) > 0 else 0.0


def pred_table_bbox_norm(tbl: dict, doc: dict):
    prov = (tbl.get("prov") or [None])[0]
    if not prov:
        return None
    pno = str(prov.get("page_no", 1))
    psize = (doc.get("pages", {}).get(pno) or {}).get("size") or {}
    pw, ph = float(psize.get("width", 1)), float(psize.get("height", 1))
    bb = prov.get("bbox") or {}
    l, r = float(bb.get("l", 0)), float(bb.get("r", 0))
    t, b = float(bb.get("t", 0)), float(bb.get("b", 0))
    if bb.get("coord_origin") == "BOTTOMLEFT":
        y0, y1 = ph - max(t, b), ph - min(t, b)
    else:
        y0, y1 = min(t, b), max(t, b)
    return (l / pw, y0 / ph, r / pw, y1 / ph)


gt_list = json.loads(GT_PATH.read_text(encoding="utf-8"))
gt_map = {Path(p["page_info"]["image_path"]).stem: p for p in gt_list}

records = []
for model in MODELS:
    m_dir = BASE_DIR / model
    for f in sorted(m_dir.glob("*.json")):
        if f.name in {"errors.json", "current_files.json"}:
            continue
        page = gt_map.get(f.stem)
        if page is None:
            continue
        pw = float(page["page_info"]["width"])
        ph = float(page["page_info"]["height"])
        gt_tbls = [
            d for d in page.get("layout_dets", [])
            if d.get("category_type") == "table" and not d.get("ignore") and (d.get("html") or "").strip()
        ]
        if not gt_tbls:
            continue
        doc = json.loads(f.read_text(encoding="utf-8"))
        pr_tbls = doc.get("tables") or []
        pr_boxes = [pred_table_bbox_norm(t, doc) for t in pr_tbls]
        used = set()
        layout_type = page.get("page_info", {}).get("page_attribute", {}).get("layout", "unknown")
        for gd in gt_tbls:
            poly = gd["poly"]
            gb = (min(poly[0::2]) / pw, min(poly[1::2]) / ph, max(poly[0::2]) / pw, max(poly[1::2]) / ph)
            best_i, best_iou = -1, 0.3
            for idx, pb in enumerate(pr_boxes):
                if idx in used or pb is None:
                    continue
                v = box_iou(gb, pb)
                if v > best_iou:
                    best_iou, best_i = v, idx
            gt_tree = html_to_tree(gd["html"])
            complexity = "Complex (Spanning)" if is_complex_tree(gt_tree) else "Simple (Grid)"
            if best_i >= 0:
                used.add(best_i)
                pr_tree = docling_table_to_tree(pr_tbls[best_i])
                teds = compute_teds(pr_tree, gt_tree, False) * 100.0
                teds_s = compute_teds(pr_tree, gt_tree, True) * 100.0
                matched = True
            else:
                teds = 0.0
                teds_s = 0.0
                matched = False
            records.append({
                "model": model,
                "file": f.stem,
                "layout": layout_type,
                "complexity": complexity,
                "matched": matched,
                "teds": teds,
                "teds_struct": teds_s,
            })

df = pd.DataFrame(records)
print(f"Evaluated {len(df) // len(MODELS)} tables across {df['file'].nunique()} pages x {len(MODELS)} models")

Evaluated 406 tables across 354 pages x 3 models


In [2]:
# Table 1: Overall Table Recognition Performance
rows = []
for model in MODELS:
    sub = df[df["model"] == model]
    m_sub = sub[sub["matched"]]
    rows.append({
        "Model": model,
        "Tables": len(sub),
        "Matched % (↑)": round(sub["matched"].mean() * 100.0, 2),
        "TEDS % (↑)": round(sub["teds"].mean(), 2),
        "TEDS-Struct % (↑)": round(sub["teds_struct"].mean(), 2),
        "TEDS Matched-only % (↑)": round(m_sub["teds"].mean(), 2) if len(m_sub) else 0.0,
        "TEDS-Struct Matched-only % (↑)": round(m_sub["teds_struct"].mean(), 2) if len(m_sub) else 0.0,
    })
overall_df = pd.DataFrame(rows).sort_values("TEDS % (↑)", ascending=False).reset_index(drop=True)
display(overall_df)

,Model,Tables,Matched % (↑),TEDS % (↑),TEDS-Struct % (↑),TEDS Matched-only % (↑),TEDS-Struct Matched-only % (↑)
0,RapidTable — UniTable,406,94.83,77.60,85.55,81.84,90.22
1,TableFormer ACCURATE,406,94.83,75.77,81.53,79.90,85.98
2,RapidTable — SLANet_plus,406,94.83,71.53,81.23,75.44,85.66


In [3]:
# Table 2: Simple (Grid) vs Complex (Spanning) Tables
comp_rows = []
for comp in ["Simple (Grid)", "Complex (Spanning)"]:
    for model in MODELS:
        sub = df[(df["complexity"] == comp) & (df["model"] == model)]
        if len(sub) == 0:
            continue
        comp_rows.append({
            "Table Type": comp,
            "Model": model,
            "Tables": len(sub),
            "TEDS % (↑)": round(sub["teds"].mean(), 2),
            "TEDS-Struct % (↑)": round(sub["teds_struct"].mean(), 2),
        })
display(pd.DataFrame(comp_rows))

,Table Type,Model,Tables,TEDS % (↑),TEDS-Struct % (↑)
0,Simple (Grid),TableFormer ACCURATE,129,85.50,89.93
1,Simple (Grid),RapidTable — SLANet_plus,129,80.33,88.21
2,Simple (Grid),RapidTable — UniTable,129,82.23,87.42
3,Complex (Spanning),TableFormer ACCURATE,277,71.23,77.62
4,Complex (Spanning),RapidTable — SLANet_plus,277,67.44,77.99
5,Complex (Spanning),RapidTable — UniTable,277,75.45,84.68


In [4]:
# Table 3: Per Japanese PDF Type (layout)
counts = df[df["model"] == MODELS[0]].groupby("layout")["teds"].count()
piv_t = df.pivot_table(index="layout", columns="model", values="teds", aggfunc="mean")[MODELS].round(2)
piv_s = df.pivot_table(index="layout", columns="model", values="teds_struct", aggfunc="mean")[MODELS].round(2)

per_type = pd.DataFrame({"Tables": counts})
for m in MODELS:
    per_type[f"{m} TEDS%"] = piv_t[m]
for m in MODELS:
    per_type[f"{m} Struct%"] = piv_s[m]
display(per_type.sort_values("Tables", ascending=False))

,Tables,TableFormer ACCURATE TEDS%,RapidTable — SLANet_plus TEDS%,RapidTable — UniTable TEDS%,TableFormer ACCURATE Struct%,RapidTable — SLANet_plus Struct%,RapidTable — UniTable Struct%
layout,,,,,,,
text_book,49,77.08,72.51,78.92,82.80,82.20,86.97
report,39,79.97,72.66,78.05,84.74,82.37,83.50
lab_notebook,36,75.24,65.98,73.30,82.60,78.43,84.80
two_column_simple,34,90.23,87.44,93.90,96.45,98.29,99.36
newspaper,32,71.25,71.03,77.30,76.96,79.24,83.86
user_manual,32,78.46,73.05,75.86,83.22,81.42,83.93
handwritten_note,30,65.64,60.41,68.31,73.03,71.57,80.12
press_release,28,78.52,75.19,76.75,82.33,84.93,85.00
slide,28,78.55,77.68,80.06,83.08,83.20,87.02
